# 🧬 BioReason-X | Mutation-to-Therapy Intelligence Playground

Welcome to the **BioReason-X Playground**! This interactive Jupyter notebook provides a complete walk-through of the BioReason-X multi-agent genomic reasoning pipeline.

BioReason-X is an explainable biomedical multi-agent system designed to assist oncology researchers and clinicians by tracing the biological causal chain of genomic alterations:
$$\text{Genomic Mutation} \rightarrow \text{Gene} \rightarrow \text{Protein Sequence Impact} \rightarrow \text{Signaling Pathway Disruption} \rightarrow \text{Disease Association} \rightarrow \text{Targeted Therapy}$$

---

### 🤖 The 7-Agent Collaborative Graph
We orchestrate 7 specialized agents using a stateful **LangGraph** workflow:
1. **Mutation Analysis Agent**: Parses variant classification and pathogenicity records (NCBI ClinVar).
2. **Gene/Protein Agent**: Evaluates structural folding shifts and affected domains (MyGene.info/UniProt).
3. **Pathway Agent**: Resolves disrupted biological networks and downstream targets (Reactome).
4. **Literature Agent**: Scrapes and indexes scientific publications (PubMed) to a local RAG vector database.
5. **Therapy Agent**: Matches FDA-approved and investigational drug agents (DGIdb).
6. **Validation Agent**: Audits and scores evidence claims (ChromaDB Vector RAG).
7. **Consensus Agent**: Aggregates all insights to synthesize a cohesive, evidence-grounded clinical report.

Let's install the dependencies and run the pipeline!

In [ ]:
# Install required dependencies
# Note: In some environments, you may need to restart the kernel after installing
!pip install -q langgraph langchain langchain-community chromadb networkx pydantic sentence-transformers requests python-dotenv plotly pillow gTTS

In [ ]:
import os
import sys
import json
import re
import math
import urllib.request
import urllib.parse
from typing import List, Optional, Dict, Any, Type, Tuple
from pydantic import BaseModel, Field
import networkx as nx
import plotly.graph_objects as go
from openai import OpenAI
from gtts import gTTS
from langgraph.graph import StateGraph, END
from IPython.display import Audio, display, Markdown

# Force pure-Python implementation for protobuf to resolve version mismatches in the environment
os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"

print("✅ Imports loaded successfully. Protobuf environment variable set.")

In [ ]:
# =====================================================================
# 1. STATE SCHEMAS DEFINITION
# =====================================================================

class MutationAnalysisState(BaseModel):
    gene_symbol: str = Field(description="The HGNC gene symbol affected (e.g. BRCA1)")
    variant_nomenclature: str = Field(description="Genomic or protein nomenclature of variation (e.g. c.5266dupC)")
    pathogenicity: str = Field(description="Pathogenicity classification (e.g. Pathogenic, Likely Pathogenic)")
    mutation_type: str = Field(description="Molecular type (e.g. Missense, Frameshift)")
    clinical_summary: str = Field(description="Short clinical overview of the variant")

class GeneProteinState(BaseModel):
    uniprot_id: str = Field(description="UniProt database identifier")
    protein_domains: str = Field(description="Affected protein domains and functional regions")
    functional_consequence: str = Field(description="Consequence of mutation on the protein's activity")
    molecular_mechanism: str = Field(description="Underlying molecular mechanism of impact")

class PathwayState(BaseModel):
    affected_pathways: List[str] = Field(default=[], description="Biological pathways disrupted or activated")
    cellular_impact: str = Field(description="Downstream cellular phenotype impact")
    downstream_targets: List[str] = Field(default=[], description="Downstream signaling target nodes")

class LiteratureState(BaseModel):
    relevant_pmids: List[str] = Field(default=[], description="PMIDs of retrieved supporting articles")
    extracted_sentences: List[str] = Field(default=[], description="Relevant text extractions from abstracts")
    evidence_strength: str = Field(description="Strength annotation (e.g. Phase III, In Vitro)")

class TherapyState(BaseModel):
    recommended_therapies: List[str] = Field(default=[], description="Matched targeted therapies")
    mechanism_of_action: str = Field(description="Molecular rationale for the drug response")
    resistance_risks: str = Field(description="Potential mechanism of secondary drug resistance")

class ValidationState(BaseModel):
    is_validated: bool = Field(description="True if validation checks pass")
    validation_score: float = Field(description="Confidence/validation score between 0 and 100")
    contradictions_found: str = Field(description="Description of any contradicting literature findings")
    evidence_rating: str = Field(description="Overall rating of evidence base (e.g., Strongly Supported)")

class ConsensusState(BaseModel):
    final_consensus_statement: str = Field(description="Explainable consensus summary")
    confidence_rating: str = Field(description="Formatted confidence score percentage")
    recommended_actions: List[str] = Field(default=[], description="Clinician next step actions")
    references: List[str] = Field(default=[], description="Formatted key references list")

class BioReasonState(BaseModel):
    mutation_query: str = Field(description="User genomic mutation input query")
    raw_bio_data: Optional[Dict[str, Any]] = Field(default=None, description="Raw fetched biomedical records")
    
    mutation_analysis: Optional[MutationAnalysisState] = Field(default=None)
    gene_protein: Optional[GeneProteinState] = Field(default=None)
    pathway: Optional[PathwayState] = Field(default=None)
    literature: Optional[LiteratureState] = Field(default=None)
    therapy: Optional[TherapyState] = Field(default=None)
    validation: Optional[ValidationState] = Field(default=None)
    consensus: Optional[ConsensusState] = Field(default=None)
    
    reasoning_trace: List[str] = Field(default=[], description="Chronological reasoning step trace list")

print("✅ State models and validation schemas defined.")

In [ ]:
# =====================================================================
# 2. UNIFIED LLM CLIENT
# =====================================================================

class LLMClient:
    """
    Unified LLM Client supporting:
    - Local vLLM servers running on AMD Instinct MI300X GPUs.
    - Fallbacks to commercial cloud models (OpenAI or Gemini).
    - An offline deterministic mock engine that generates rich responses when keys/servers are offline.
    """
    
    def __init__(self):
        self.provider = os.getenv("LLM_PROVIDER", "mock").lower()
        self.text_api_base = os.getenv("VLLM_TEXT_API_BASE", "http://localhost:8000/v1")
        self.text_model = os.getenv("VLLM_TEXT_MODEL", "Qwen/Qwen2.5-14B-Instruct-AWQ")
        
        self.openai_key = os.getenv("OPENAI_API_KEY", "")
        self.openai_model = os.getenv("OPENAI_API_MODEL", "gpt-4o")
        
        # Initialize client if keys/endpoints exist
        self.client = None
        if self.provider == "vllm":
            self.client = OpenAI(base_url=self.text_api_base, api_key="dummy-key")
        elif self.provider == "openai" and self.openai_key:
            self.client = OpenAI(api_key=self.openai_key)
        else:
            print("LLM Client: No active model servers or API keys detected. Initializing offline mock processor.")
            self.provider = "mock"

    def generate_text(self, prompt: str, system_prompt: str = "You are a professional clinical geneticist.") -> str:
        """Generates plain-text completion."""
        if self.provider == "mock" or not self.client:
            return self._mock_text_generation(prompt)
            
        try:
            response = self.client.chat.completions.create(
                model=self.text_model if self.provider == "vllm" else self.openai_model,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": prompt}
                ],
                temperature=0.1,
                max_tokens=1500
            )
            return response.choices[0].message.content
        except Exception as e:
            print(f"LLM API execution error: {e}. Falling back to offline generation...")
            return self._mock_text_generation(prompt)

    def generate_json(self, prompt: str, system_prompt: str, response_schema: Type[BaseModel]) -> BaseModel:
        """
        Generates structured outputs matching a Pydantic model.
        Falls back to local parser if endpoints are unavailable.
        """
        if self.provider == "mock" or not self.client:
            return self._mock_json_generation(prompt, response_schema)
            
        try:
            response = self.client.chat.completions.create(
                model=self.text_model if self.provider == "vllm" else self.openai_model,
                messages=[
                    {"role": "system", "content": f"{system_prompt}\nYou MUST output valid raw JSON matching this schema: {response_schema.model_json_schema()}"},
                    {"role": "user", "content": prompt}
                ],
                response_format={"type": "json_object"},
                temperature=0.0,
                max_tokens=1000
            )
            
            content = response.choices[0].message.content
            parsed_data = json.loads(content)
            return response_schema.model_validate(parsed_data)
        except Exception as e:
            print(f"LLM JSON parsing error: {e}. Falling back to deterministic structured parser...")
            return self._mock_json_generation(prompt, response_schema)

    def _mock_text_generation(self, prompt: str) -> str:
        """Helper to create detailed natural language reasoning traces offline."""
        if "consensus" in prompt.lower() or "final report" in prompt.lower():
            return "Consensus Statement: Analysis reveals high confidence target matches. " \
                   "The genomic alteration destabilizes binding, indicating PARP inhibitor sensitivity."
        return "Offline Analysis: Active pocket configuration resolved. Significant hydrogen bonding shift identified."

    def _mock_json_generation(self, prompt: str, schema: Type[BaseModel]) -> BaseModel:
        """Generates deterministic Pydantic objects based on regex keyword matching."""
        prompt_lower = prompt.lower()
        schema_name = schema.__name__
        
        if schema_name == "MutationAnalysisState":
            gene = "BRCA1"
            mut = "c.5266dupC"
            patho = "Pathogenic"
            m_type = "Frameshift Insertion"
            
            if "egfr" in prompt_lower:
                gene = "EGFR"
                mut = "L858R"
                m_type = "Missense Substitution"
            elif "braf" in prompt_lower:
                gene = "BRAF"
                mut = "V600E"
                m_type = "Missense Substitution"
            elif "kras" in prompt_lower:
                gene = "KRAS"
                mut = "G12C"
                m_type = "Missense Substitution"
            elif "alk" in prompt_lower:
                gene = "ALK"
                mut = "F1174L"
                patho = "Pathogenic / Resistant"
                m_type = "Missense Substitution"
                
            return schema(
                gene_symbol=gene,
                variant_nomenclature=mut,
                pathogenicity=patho,
                mutation_type=m_type,
                clinical_summary=f"Parsed {gene} variant showing pathogenic significance."
            )
            
        elif schema_name == "GeneProteinState":
            gene = "BRCA1"
            u_id = "P38398"
            conseq = "Truncates C-terminal BRCT domain, damaging DNA double-strand repair."
            if "egfr" in prompt_lower:
                gene = "EGFR"
                u_id = "P00533"
                conseq = "Locks kinase domain in active state promoting MAPK transcription."
            elif "braf" in prompt_lower:
                gene = "BRAF"
                u_id = "P15056"
                conseq = "Keeps kinase active as monomer activating downstream MEK."
            elif "kras" in prompt_lower:
                gene = "KRAS"
                u_id = "P01116"
                conseq = "Impairs intrinsic GTP hydrolysis, locking KRAS in active GTP state."
            elif "alk" in prompt_lower:
                gene = "ALK"
                u_id = "Q9UM73"
                conseq = "Increases ATP binding affinity, bypassing first-generation TKIs."
                
            return schema(
                uniprot_id=u_id,
                protein_domains="Kinase catalytic domain or repair interaction domain",
                functional_consequence=conseq,
                molecular_mechanism="Constitutive downstream cascade activation or repair loop truncation"
            )
            
        elif schema_name == "PathwayState":
            pathways = ["DNA Double-Strand Break Repair", "Homologous Recombination"]
            desc = "Inability to repair double-strand breaks leads to genomic instability."
            if "egfr" in prompt_lower:
                pathways = ["EGFR Signaling Pathway", "MAPK Cascade"]
                desc = "Upregulation of cell cycle progression and survival cascades."
            elif "braf" in prompt_lower:
                pathways = ["RAF-MAPK Cascade"]
                desc = "Continuous cell growth signaling bypassing receptor control."
            elif "kras" in prompt_lower:
                pathways = ["Signaling by RAS variants", "MAPK Pathway"]
                desc = "Continuous activation of RAF-MEK-ERK growth signaling cascades."
            elif "alk" in prompt_lower:
                pathways = ["Signaling by ALK", "AKT Pathway"]
                desc = "Upregulation of survival pathways causing drug-resistant expansion."
                
            return schema(
                affected_pathways=pathways,
                cellular_impact=desc,
                downstream_targets=["RAF", "MEK", "ERK", "AKT"]
            )
            
        elif schema_name == "LiteratureState":
            return schema(
                relevant_pmids=["29333925", "20729471"],
                extracted_sentences=[
                    "Evidence matches clinical efficacy metrics.",
                    "Significantly increased progression-free survival rates recorded."
                ],
                evidence_strength="High (Phase III Trials)"
            )
            
        elif schema_name == "TherapyState":
            drugs = ["Olaparib", "Talazoparib"]
            rat = "Induces synthetic lethality in HR-deficient cells."
            if "egfr" in prompt_lower:
                drugs = ["Osimertinib", "Erlotinib"]
                rat = "Competes with ATP binding in active site pockets."
            elif "braf" in prompt_lower:
                drugs = ["Vemurafenib", "Dabrafenib"]
                rat = "Binds to monomeric active BRAF kinase pocket."
            elif "kras" in prompt_lower:
                drugs = ["Sotorasib", "Adagrasib"]
                rat = "Covalently links to Cysteine at position 12, locking GDP state."
            elif "alk" in prompt_lower:
                drugs = ["Lorlatinib"]
                rat = "Macrocyclic inhibitor overcoming secondary pocket ATP-binding shifts."
                
            return schema(
                recommended_therapies=drugs,
                mechanism_of_action=rat,
                resistance_risks="Acquisition of secondary kinase domain gatekeeper variants."
            )
            
        elif schema_name == "ValidationState":
            return schema(
                is_validated=True,
                validation_score=92.5,
                contradictions_found="None identified in current vector references.",
                evidence_rating="Strongly Supported"
            )
            
        elif schema_name == "ConsensusState":
            return schema(
                final_consensus_statement="BioReason-X concludes the identified variant is pathogenic. Targeted therapeutic strategies (inhibitors/PARPi) are highly indicated based on clinical trial evidence.",
                confidence_rating="92%",
                recommended_actions=["Initiate clinical sequencing confirm", "Select matched therapy options"],
                references=["Ledermann J, et al. Lancet Oncol. 2010"]
            )
            
        return schema.model_construct()

print("✅ LLM Client instantiated.")

In [ ]:
# =====================================================================
# 3. BIOMEDICAL DATA FETCHER
# =====================================================================

class DataFetcher:
    """
    Handles live querying of real-world biological databases:
    - NCBI ClinVar (via E-Utilities API)
    - NCBI PubMed (via E-Utilities API)
    - MyGene.info API (gene annotations, pathways, UniProt)
    - DGIdb API (Drug-Gene Interaction Database)
    Falls back to a curated local database file or inline mock dictionary.
    """
    
    def __init__(self, cache_file_path: str = "data/cached_mutations.json"):
        self.cache_file_path = cache_file_path
        self.local_cache = {}
        self._load_cache()

    def _load_cache(self):
        """Loads local curated database or uses inline fallback."""
        try:
            if os.path.exists(self.cache_file_path):
                with open(self.cache_file_path, "r", encoding="utf-8") as f:
                    self.local_cache = json.load(f)
                print("Loaded cache from local file.")
            else:
                alt_path = os.path.join(os.path.dirname(os.path.abspath('')), "data/cached_mutations.json")
                if os.path.exists(alt_path):
                    with open(alt_path, "r", encoding="utf-8") as f:
                        self.local_cache = json.load(f)
                    print(f"Loaded cache from alternative path: {alt_path}")
                else:
                    print("Cache file not found. Initializing built-in backup mock dictionary.")
                    self._load_fallback_cache()
        except Exception as e:
            print(f"Error loading cache: {e}. Loading backup mock data...")
            self._load_fallback_cache()

    def _load_fallback_cache(self):
        self.local_cache = {
            "BRCA1 c.5266dupC": {
                "gene": "BRCA1",
                "mutation": "c.5266dupC (p.Gln1756ProfsTer74)",
                "pathogenicity": "Pathogenic",
                "mutation_type": "Frameshift Insertion",
                "chromosome": "Chr 17: 43044295",
                "review_status": "practice guideline (4 stars)",
                "clinical_significance": "Pathogenic variant associated with breast/ovarian cancer.",
                "protein_impact": {
                    "uniprot_id": "P38398",
                    "consequence": "Truncates the C-terminal BRCT domain essential for DNA repair.",
                    "residues_affected": "1756-1863 (truncated)"
                },
                "pathways": [
                    {"id": "R-HSA-5685939", "name": "Homologous Recombination Repair"},
                    {"id": "R-HSA-73894", "name": "DNA Double-Strand Break Repair"}
                ],
                "drugs": [
                    {"name": "Olaparib", "type": "inhibitor", "indication": "FDA Approved", "rationale": "Induces synthetic lethality in BRCA-mutant cells."},
                    {"name": "Talazoparib", "type": "inhibitor", "indication": "FDA Approved", "rationale": "Potent PARP inhibitor and trapper."}
                ],
                "pubmed": [
                    {"pmid": "29333925", "title": "Talazoparib in BRCA1/2-Mutated Breast Cancer", "journal": "N Engl J Med", "year": "2018", "authors": "Litton JK, et al.", "abstract": "Talazoparib provides significant progression-free survival benefit over chemotherapy."}
                ]
            },
            "EGFR L858R": {
                "gene": "EGFR",
                "mutation": "L858R (c.2573T>G)",
                "pathogenicity": "Pathogenic / Drug Sensitive",
                "mutation_type": "Missense Substitution",
                "chromosome": "Chr 7: 55191822",
                "review_status": "reviewed by expert panel (3 stars)",
                "clinical_significance": "Pathogenic variant associated with NSCLC and responsiveness to TKIs.",
                "protein_impact": {
                    "uniprot_id": "P00533",
                    "consequence": "Locks kinase domain in a constitutively active state promoting continuous growth cascades.",
                    "residues_affected": "858 (kinase domain)"
                },
                "pathways": [
                    {"id": "R-HSA-177929", "name": "Signaling by EGFR"},
                    {"id": "R-HSA-5684996", "name": "MAPK family signaling cascades"}
                ],
                "drugs": [
                    {"name": "Osimertinib", "type": "inhibitor", "indication": "FDA Approved", "rationale": "Third-generation irreversible EGFR TKI."},
                    {"name": "Erlotinib", "type": "inhibitor", "indication": "FDA Approved", "rationale": "First-generation reversible EGFR TKI."}
                ],
                "pubmed": [
                    {"pmid": "29151359", "title": "Osimertinib in Untreated EGFR-Mutated NSCLC", "journal": "N Engl J Med", "year": "2018", "authors": "Soria JC, et al.", "abstract": "Osimertinib showed efficacy superior to standard TKIs in L858R mutations."}
                ]
            }
        }

    def get_mutation_data(self, query: str) -> Dict[str, Any]:
        query_clean = query.strip()
        for key, value in self.local_cache.items():
            if query_clean.lower() in key.lower() or key.lower() in query_clean.lower():
                return value
                
        gene_symbol = self._extract_gene_symbol(query_clean)
        print(f"⚠️ Query '{query_clean}' not in local database. Fetching from live public databases...")
        return self._fetch_live_data(query_clean, gene_symbol)

    def _extract_gene_symbol(self, query: str) -> str:
        tokens = query.split()
        if tokens:
            symbol = tokens[0].upper()
            symbol = re.sub(r'[^A-Z0-9]', '', symbol)
            return symbol
        return "UNKNOWN"

    def _fetch_live_data(self, query: str, gene_symbol: str) -> Dict[str, Any]:
        data = {
            "gene": gene_symbol,
            "mutation": query,
            "pathogenicity": "Uncertain Significance",
            "mutation_type": "Unknown",
            "chromosome": "Unknown",
            "review_status": "none",
            "clinical_significance": "Dynamic Live API Search Results.",
            "protein_impact": {
                "uniprot_id": "Unknown",
                "consequence": "No sequence structure consequence found in ClinVar. Analysis generated via LLM reasoning.",
                "residues_affected": "Unknown"
            },
            "pathways": [],
            "drugs": [],
            "pubmed": []
        }

        clinvar_info = self._query_clinvar(query)
        if clinvar_info:
            data.update(clinvar_info)

        mygene_info = self._query_mygene(gene_symbol)
        if mygene_info:
            if "protein_impact" in mygene_info:
                data["protein_impact"].update(mygene_info["protein_impact"])
            if "pathways" in mygene_info:
                data["pathways"] = mygene_info["pathways"]
            if "gene" in mygene_info:
                data["gene"] = mygene_info["gene"]

        drugs_list = self._query_dgidb(gene_symbol)
        if drugs_list:
            data["drugs"] = drugs_list
        else:
            data["drugs"] = [{
                "name": f"Targeted {gene_symbol} inhibitor",
                "type": "inhibitor",
                "indication": "Investigational / Research Target",
                "rationale": f"Identified as a genomic target for {gene_symbol} binding."
            }]

        pubmed_articles = self._query_pubmed(query)
        if pubmed_articles:
            data["pubmed"] = pubmed_articles
            
        return data

    def _query_clinvar(self, term: str) -> Optional[Dict[str, Any]]:
        try:
            encoded_term = urllib.parse.quote(term)
            search_url = f"https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi?db=clinvar&term={encoded_term}&retmode=json"
            req = urllib.request.Request(search_url, headers={'User-Agent': 'Mozilla/5.0'})
            with urllib.request.urlopen(req, timeout=5) as response:
                result = json.loads(response.read().decode('utf-8'))
                id_list = result.get("esearchresult", {}).get("idlist", [])
            if not id_list:
                return None
            ids = ",".join(id_list[:3])
            summary_url = f"https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esummary.fcgi?db=clinvar&id={ids}&retmode=json"
            req_sum = urllib.request.Request(summary_url, headers={'User-Agent': 'Mozilla/5.0'})
            with urllib.request.urlopen(req_sum, timeout=5) as response:
                summary_result = json.loads(response.read().decode('utf-8'))
                uid_data = summary_result.get("result", {})
            for uid in id_list:
                record = uid_data.get(uid, {})
                title = record.get("title", "")
                clinical_signif = record.get("clinical_significance", {}).get("description", "Uncertain Significance")
                mut_type = record.get("variant_type", "Unknown")
                loc = record.get("variation_loc", [{}])[0]
                chrom = f"Chr {loc.get('chr', 'Unknown')}: {loc.get('start', 'Unknown')}"
                return {
                    "pathogenicity": clinical_signif,
                    "mutation_type": mut_type,
                    "chromosome": chrom,
                    "review_status": record.get("clinical_significance", {}).get("review_status", "none"),
                    "clinical_significance": f"Variant '{title}' is listed in ClinVar with significance: {clinical_signif}."
                }
        except Exception as e:
            print(f"ClinVar API query failed: {e}")
        return None

    def _query_mygene(self, gene_symbol: str) -> Optional[Dict[str, Any]]:
        try:
            url = f"https://mygene.info/v3/query?q=symbol:{gene_symbol}&fields=name,summary,pathway,uniprot&retmode=json"
            req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
            with urllib.request.urlopen(req, timeout=5) as response:
                result = json.loads(response.read().decode('utf-8'))
                hits = result.get("hits", [])
            if not hits:
                return None
            hit = hits[0]
            summary = hit.get("summary", "No official summary description available.")
            uniprot_data = hit.get("uniprot", {})
            uniprot_id = "Unknown"
            if isinstance(uniprot_data, dict):
                uniprot_id = uniprot_data.get("Swiss-Prot", "Unknown")
            elif isinstance(uniprot_data, list) and len(uniprot_data) > 0:
                uniprot_id = uniprot_data[0]
            pathways_list = []
            pathways_data = hit.get("pathway", {})
            if isinstance(pathways_data, dict) and "reactome" in pathways_data:
                reactome = pathways_data["reactome"]
                if isinstance(reactome, dict):
                    reactome = [reactome]
                for p in reactome[:3]:
                    pathways_list.append({"id": p.get("id", "Unknown"), "name": p.get("name", "Unknown Pathway")})
            return {
                "gene": hit.get("symbol", gene_symbol),
                "protein_impact": {"uniprot_id": uniprot_id, "consequence": summary},
                "pathways": pathways_list
            }
        except Exception as e:
            print(f"MyGene API query failed: {e}")
        return None

    def _query_dgidb(self, gene_symbol: str) -> List[Dict[str, Any]]:
        try:
            url = f"https://dgidb.org/api/v2/interactions.json?genes={gene_symbol}"
            req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
            with urllib.request.urlopen(req, timeout=5) as response:
                result = json.loads(response.read().decode('utf-8'))
                matched_genes = result.get("matchedTerms", [])
            drugs = []
            seen_drugs = set()
            for mg in matched_genes:
                interactions = mg.get("interactions", [])
                for inter in interactions[:5]:
                    drug_name = inter.get("drugName", "").capitalize()
                    interaction_type = inter.get("interactionTypes", ["modulator"])[0]
                    if drug_name and drug_name not in seen_drugs:
                        seen_drugs.add(drug_name)
                        drugs.append({
                            "name": drug_name,
                            "type": interaction_type,
                            "indication": "Interacting Agent (DGIdb)",
                            "rationale": f"Identified as an FDA-approved or investigational {interaction_type} targeting {gene_symbol}."
                        })
            return drugs
        except Exception as e:
            print(f"DGIdb API query failed: {e}")
        return []

    def _query_pubmed(self, term: str) -> List[Dict[str, Any]]:
        try:
            query = f"{term} AND (therapy OR drug resistance OR clinical trial)"
            encoded_query = urllib.parse.quote(query)
            search_url = f"https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi?db=pubmed&term={encoded_query}&retmode=json"
            req = urllib.request.Request(search_url, headers={'User-Agent': 'Mozilla/5.0'})
            with urllib.request.urlopen(req, timeout=5) as response:
                result = json.loads(response.read().decode('utf-8'))
                id_list = result.get("esearchresult", {}).get("idlist", [])
            if not id_list:
                return []
            ids = ",".join(id_list[:3])
            summary_url = f"https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esummary.fcgi?db=pubmed&id={ids}&retmode=json"
            req_sum = urllib.request.Request(summary_url, headers={'User-Agent': 'Mozilla/5.0'})
            with urllib.request.urlopen(req_sum, timeout=5) as response:
                summary_result = json.loads(response.read().decode('utf-8'))
                uid_data = summary_result.get("result", {})
            articles = []
            for uid in id_list[:3]:
                record = uid_data.get(uid, {})
                title = record.get("title", "No Title")
                authors = ", ".join([a.get("name", "") for a in record.get("authors", [])[:3]]) + ", et al."
                pub_date = record.get("pubdate", "Unknown")
                year = pub_date.split()[0] if pub_date else "Unknown"
                source = record.get("source", "PubMed")
                snippet = f"This publication in {source} details the clinical significance of {term} mutations in oncology studies."
                articles.append({
                    "pmid": uid, "title": title, "journal": source, "year": year,
                    "authors": authors, "abstract": snippet
                })
            return articles
        except Exception as e:
            print(f"PubMed API query failed: {e}")
        return []

print("✅ Data Fetcher service registered.")

In [ ]:
# =====================================================================
# 4. CHROMADB VECTOR RAG STORE
# =====================================================================

class BiomedicalVectorStore:
    """
    RAG vector store wrapper using ChromaDB for persistent storage
    and a robust, local pure-Python TF-IDF text similarity engine
    to bypass Hugging Face Hub network dependencies, timeouts, and rate limits.
    """
    
    def __init__(self, persist_dir: str = "data/chroma_db"):
        self.persist_dir = persist_dir
        os.makedirs(self.persist_dir, exist_ok=True)
        
        try:
            self.client = chromadb.PersistentClient(path=self.persist_dir)
        except Exception:
            self.client = chromadb.EphemeralClient()
            
        self.collection = self.client.get_or_create_collection(
            name="biomedical_literature",
            metadata={"hnsw:space": "cosine"}
        )

    def _tokenize(self, text: str) -> List[str]:
        return re.findall(r'[a-zA-Z0-9_]+', text.lower())

    def _compute_cosine_similarity(self, query: str, doc: str) -> float:
        q_tokens = self._tokenize(query)
        d_tokens = self._tokenize(doc)
        
        if not q_tokens or not d_tokens:
            return 0.0
            
        q_freq = {}
        for token in q_tokens:
            q_freq[token] = q_freq.get(token, 0) + 1
            
        d_freq = {}
        for token in d_tokens:
            d_freq[token] = d_freq.get(token, 0) + 1
            
        dot_product = 0.0
        for token, freq in q_freq.items():
            if token in d_freq:
                dot_product += freq * d_freq[token]
                
        q_norm = math.sqrt(sum(f ** 2 for f in q_freq.values()))
        d_norm = math.sqrt(sum(f ** 2 for f in d_freq.values()))
        
        if q_norm == 0 or d_norm == 0:
            return 0.0
            
        return dot_product / (q_norm * d_norm)

    def add_publications(self, publications: List[Dict[str, Any]], mutation_name: str):
        if not publications:
            return

        documents = []
        metadatas = []
        ids = []
        
        for idx, pub in enumerate(publications):
            pmid = pub.get("pmid", f"gen_{idx}")
            title = pub.get("title", "Unknown Publication")
            abstract = pub.get("abstract", "")
            journal = pub.get("journal", "Unknown Journal")
            year = pub.get("year", "Unknown Year")
            authors = pub.get("authors", "Unknown Authors")
            
            if not abstract:
                continue
                
            sentences = [s.strip() for s in abstract.split(".") if s.strip()]
            for s_idx, sentence in enumerate(sentences):
                doc_id = f"{mutation_name}_{pmid}_{s_idx}"
                documents.append(sentence)
                metadatas.append({
                    "pmid": pmid, "title": title, "journal": journal, "year": str(year), "authors": authors,
                    "mutation": mutation_name,
                    "full_source": f"{authors} ({year}). {title}. {journal}. PMID: {pmid}"
                })
                ids.append(doc_id)
                
        if documents:
            dummy_embeddings = [[0.1] * 128 for _ in range(len(documents))]
            self.collection.upsert(
                embeddings=dummy_embeddings,
                documents=documents,
                metadatas=metadatas,
                ids=ids
            )

    def search_evidence(self, query: str, limit: int = 4) -> List[Dict[str, Any]]:
        results = self.collection.get(include=["documents", "metadatas"])
        hits = []
        if results and results.get("documents"):
            docs = results["documents"]
            metas = results["metadatas"]
            for i in range(len(docs)):
                doc_text = docs[i]
                meta = metas[i]
                similarity = self._compute_cosine_similarity(query, doc_text)
                if similarity > 0.05:
                    hits.append({
                        "text": doc_text,
                        "metadata": meta,
                        "similarity": round(similarity, 3),
                        "citation": meta.get("full_source", "Unknown Citation")
                    })
                    
        hits = sorted(hits, key=lambda x: x["similarity"], reverse=True)
        return hits[:limit]

print("✅ ChromaDB Vector RAG Store service registered.")

In [ ]:
# =====================================================================
# 5. DYNAMIC RELATION KNOWLEDGE GRAPH
# =====================================================================

class BiomedicalKnowledgeGraph:
    """
    Manages the NetworkX relational knowledge graph.
    Links genomic variations through cellular pathway nodes to target therapies.
    """
    
    def __init__(self):
        self.graph = nx.DiGraph()

    def build_from_mutation_data(self, mut_name: str, data: Dict[str, Any]):
        gene_name = data.get("gene", "UnknownGene")
        protein_impact = data.get("protein_impact", {})
        uniprot_id = protein_impact.get("uniprot_id", "UnknownProt")
        
        self.graph.add_node(mut_name, type="Mutation", label=mut_name, color="#E11D48")
        self.graph.add_node(gene_name, type="Gene", label=gene_name, color="#EA580C")
        self.graph.add_node(uniprot_id, type="Protein", label=f"Protein ({uniprot_id})", color="#D97706")
        
        self.graph.add_edge(mut_name, gene_name, relation="affects")
        self.graph.add_edge(gene_name, uniprot_id, relation="encodes")
        
        for pathway in data.get("pathways", []):
            p_id = pathway.get("id", "UnknownPath")
            p_name = pathway.get("name", "Unknown Pathway")
            self.graph.add_node(p_id, type="Pathway", label=p_name, color="#0D9488")
            self.graph.add_edge(uniprot_id, p_id, relation="participates_in")
            
            disease_name = "Oncology Disease Complex"
            if "homologous" in p_name.lower() or "brca" in mut_name.lower():
                disease_name = "Breast / Ovarian Cancer"
            elif "egfr" in mut_name.lower():
                disease_name = "Non-Small Cell Lung Cancer (NSCLC)"
            elif "braf" in mut_name.lower():
                disease_name = "Malignant Melanoma"
            elif "kras" in mut_name.lower():
                disease_name = "Colorectal / Lung Adenocarcinoma"
            elif "alk" in mut_name.lower():
                disease_name = "Neuroblastoma / NSCLC"
                
            self.graph.add_node(disease_name, type="Disease", label=disease_name, color="#4F46E5")
            self.graph.add_edge(p_id, disease_name, relation="associated_with")
            
        for drug in data.get("drugs", []):
            drug_name = drug.get("name", "Unknown Drug")
            drug_type = drug.get("type", "inhibitor")
            self.graph.add_node(drug_name, type="Drug", label=f"{drug_name} ({drug_type})", color="#0891B2")
            self.graph.add_edge(drug_name, uniprot_id, relation="targets")

    def generate_plotly_figure(self) -> go.Figure:
        if len(self.graph) == 0:
            fig = go.Figure()
            fig.update_layout(title="No nodes in Knowledge Graph")
            return fig

        pos = nx.spring_layout(self.graph, k=1.0, iterations=50, seed=42)
        
        edge_x = []
        edge_y = []
        for edge in self.graph.edges():
            x0, y0 = pos[edge[0]]
            x1, y1 = pos[edge[1]]
            edge_x.extend([x0, x1, None])
            edge_y.extend([y0, y1, None])
            
        edge_trace = go.Scatter(
            x=edge_x, y=edge_y,
            line=dict(width=1, color='#475569'),
            hoverinfo='none',
            mode='lines'
        )
        
        node_x = []
        node_y = []
        node_text = []
        node_colors = []
        node_sizes = []
        
        for node in self.graph.nodes():
            x, y = pos[node]
            node_x.append(x)
            node_y.append(y)
            
            attrs = self.graph.nodes[node]
            n_type = attrs.get("type", "Unknown")
            n_label = attrs.get("label", node)
            color = attrs.get("color", "#64748B")
            
            node_colors.append(color)
            node_text.append(f"<b>Type:</b> {n_type}<br><b>Entity:</b> {n_label}")
            
            size_map = {"Mutation": 26, "Gene": 22, "Protein": 22, "Pathway": 20, "Disease": 24, "Drug": 24}
            node_sizes.append(size_map.get(n_type, 18))
            
        node_trace = go.Scatter(
            x=node_x, y=node_y,
            mode='markers+text',
            hoverinfo='text',
            text=[self.graph.nodes[node].get("label", node).split(" (")[0] for node in self.graph.nodes()],
            textposition="top center",
            textfont=dict(size=10, color='#E2E8F0'),
            marker=dict(
                showscale=False,
                color=node_colors,
                size=node_sizes,
                line=dict(width=2, color='#1E293B')
            )
        )
        node_trace.hovertext = node_text
        
        fig = go.Figure(
            data=[edge_trace, node_trace],
            layout=go.Layout(
                showlegend=False,
                hovermode='closest',
                margin=dict(b=10, l=10, r=10, t=30),
                xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
                yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
                paper_bgcolor='#0B0F19',
                plot_bgcolor='#0B0F19',
                dragmode='pan',
                title=dict(text="🧬 Relational BioReason-X Causal Biological Graph", font=dict(color="#E2E8F0", size=14))
            )
        )
        return fig

print("✅ NetworkX + Plotly Knowledge Graph service registered.")

In [ ]:
# =====================================================================
# 6. BIOMEDICAL AGENTS IMPLEMENTATION
# =====================================================================

class BiomedicalAgents:
    """
    Implements the 7 collaborative reasoning agents.
    Each agent takes the global state, processes it using the LLM Client and APIs,
    updates the state, and appends a step to the reasoning trace.
    """
    
    def __init__(self, llm_client: LLMClient, data_fetcher: DataFetcher, 
                 graph: BiomedicalKnowledgeGraph, vector_store: BiomedicalVectorStore):
        self.llm = llm_client
        self.fetcher = data_fetcher
        self.graph = graph
        self.vector_store = vector_store

    def run_mutation_analysis(self, state: BioReasonState) -> Dict[str, Any]:
        raw_data = self.fetcher.get_mutation_data(state.mutation_query)
        prompt = f"""
        Analyze this genomic mutation using the fetched database records:
        Query: {state.mutation_query}
        ClinVar Pathogenicity: {raw_data.get('pathogenicity')}
        Mutation Type: {raw_data.get('mutation_type')}
        Chromosome Location: {raw_data.get('chromosome')}
        Clinical Description: {raw_data.get('clinical_significance')}
        
        Extract gene symbol, nomenclature representation, variant type, pathogenicity, and generate a brief clinical summary.
        """
        system_prompt = "You are an expert clinical genomic analyzer. Perform variant curation."
        analysis = self.llm.generate_json(prompt, system_prompt, MutationAnalysisState)
        self.graph.build_from_mutation_data(state.mutation_query, raw_data)
        trace = f"Step 1 (Mutation Agent): Mutation parsed as {analysis.mutation_type} in gene {analysis.gene_symbol}. Classification: {analysis.pathogenicity}."
        return {
            "raw_bio_data": raw_data,
            "mutation_analysis": analysis,
            "reasoning_trace": state.reasoning_trace + [trace]
        }

    def run_gene_protein_impact(self, state: BioReasonState) -> Dict[str, Any]:
        raw_data = state.raw_bio_data or {}
        analysis = state.mutation_analysis
        prompt = f"""
        Detail the molecular and functional protein impact of this mutation:
        Gene: {analysis.gene_symbol} ({analysis.clinical_summary})
        UniProt/RefSeq info: {raw_data.get('protein_impact', {}).get('consequence')}
        UniProt ID: {raw_data.get('protein_impact', {}).get('uniprot_id')}
        
        Identify the UniProt ID, affected structural domains, functional consequence, and molecular mechanism.
        """
        system_prompt = "You are a biochemical sequence and structural biology agent."
        protein_state = self.llm.generate_json(prompt, system_prompt, GeneProteinState)
        trace = f"Step 2 (Protein Agent): UniProt ID {protein_state.uniprot_id} mapped. Molecular consequence: {protein_state.functional_consequence}."
        return {
            "gene_protein": protein_state,
            "reasoning_trace": state.reasoning_trace + [trace]
        }

    def run_pathway_analysis(self, state: BioReasonState) -> Dict[str, Any]:
        raw_data = state.raw_bio_data or {}
        gene_symbol = state.mutation_analysis.gene_symbol
        pathways_text = "\n".join([f"- {p.get('name')} (ID: {p.get('id')})" for p in raw_data.get('pathways', [])])
        prompt = f"""
        Analyze the signaling pathway disruption caused by mutation in gene {gene_symbol}.
        Linked Reactome Pathways:
        {pathways_text}
        
        Determine the affected pathways, cellular phenotype impact, and downstream effector targets.
        """
        system_prompt = "You are a systems biology and cellular pathway analysis agent."
        pathway_state = self.llm.generate_json(prompt, system_prompt, PathwayState)
        trace = f"Step 3 (Pathway Agent): Pathway analysis completed. Affected: {', '.join(pathway_state.affected_pathways)}. Cellular impact: {pathway_state.cellular_impact}."
        return {
            "pathway": pathway_state,
            "reasoning_trace": state.reasoning_trace + [trace]
        }

    def run_literature_intelligence(self, state: BioReasonState) -> Dict[str, Any]:
        raw_data = state.raw_bio_data or {}
        pubmed_articles = raw_data.get("pubmed", [])
        
        self.vector_store.add_publications(pubmed_articles, state.mutation_query)
        query = f"{state.mutation_analysis.gene_symbol} {state.mutation_analysis.variant_nomenclature} therapeutic response survival"
        hits = self.vector_store.search_evidence(query, limit=3)
        
        pmids = [hit["metadata"]["pmid"] for hit in hits]
        sentences = [f"\"{hit['text']}\" (PMID: {hit['metadata']['pmid']})" for hit in hits]
        
        prompt = f"""
        Review the following evidence extractions for mutation {state.mutation_query}:
        Extractions:
        {chr(10).join(sentences)}
        
        Verify the strength of the clinical evidence supporting therapeutic efficacy or resistance markers.
        """
        system_prompt = "You are a scientific literature intelligence agent."
        lit_state = self.llm.generate_json(prompt, system_prompt, LiteratureState)
        lit_state.relevant_pmids = list(set(pmids))
        lit_state.extracted_sentences = sentences
        trace = f"Step 4 (Literature Agent): Indexed {len(pubmed_articles)} publications in ChromaDB. Retrieved {len(sentences)} clinical evidence sentences."
        return {
            "literature": lit_state,
            "reasoning_trace": state.reasoning_trace + [trace]
        }

    def run_therapy_matching(self, state: BioReasonState) -> Dict[str, Any]:
        raw_data = state.raw_bio_data or {}
        gene_symbol = state.mutation_analysis.gene_symbol
        drugs_data = raw_data.get("drugs", [])
        drugs_text = "\n".join([f"- {d.get('name')} ({d.get('type')}): {d.get('indication')}" for d in drugs_data])
        prompt = f"""
        Identify targeted therapies for a mutation in gene {gene_symbol}.
        DGIdb Drug Target Interactions:
        {drugs_text}
        
        Literature evidence extractions:
        {state.literature.extracted_sentences}
        
        Match specific drug names, explain their molecular mechanism, and list resistance risks.
        """
        system_prompt = "You are a clinical pharmacogenomics agent."
        therapy_state = self.llm.generate_json(prompt, system_prompt, TherapyState)
        trace = f"Step 5 (Therapy Agent): Identified {len(therapy_state.recommended_therapies)} matched therapies: {', '.join(therapy_state.recommended_therapies)}."
        return {
            "therapy": therapy_state,
            "reasoning_trace": state.reasoning_trace + [trace]
        }

    def run_evidence_validation(self, state: BioReasonState) -> Dict[str, Any]:
        claims = f"Therapies: {state.therapy.recommended_therapies}. Rationale: {state.therapy.mechanism_of_action}."
        validation_hits = self.vector_store.search_evidence(claims, limit=3)
        valid_text = "\n".join([f"- {h['text']} (Similarity: {h['similarity']})" for h in validation_hits])
        prompt = f"""
        Perform a validation audit on these claims against the retrieved literature context.
        Claims: {claims}
        Retrieved Context:
        {valid_text}
        
        Check for errors or contradictions. Score validation confidence (0-100), identify contradictions, and rate evidence base.
        """
        system_prompt = "You are a medical evidence review board agent."
        validation_state = self.llm.generate_json(prompt, system_prompt, ValidationState)
        trace = f"Step 6 (Validation Agent): Evidence validated with confidence score: {validation_state.validation_score}%. Rating: {validation_state.evidence_rating}."
        return {
            "validation": validation_state,
            "reasoning_trace": state.reasoning_trace + [trace]
        }

    def run_consensus_aggregation(self, state: BioReasonState) -> Dict[str, Any]:
        prompt = f"""
        Synthesize the final report for mutation {state.mutation_query}:
        - Analysis: {state.mutation_analysis.clinical_summary} ({state.mutation_analysis.pathogenicity})
        - Protein Impact: {state.gene_protein.functional_consequence} ({state.gene_protein.molecular_mechanism})
        - Cellular Pathways: {state.pathway.affected_pathways}
        - Matched Drugs: {state.therapy.recommended_therapies} (Mechanism: {state.therapy.mechanism_of_action})
        - Validation Score: {state.validation.validation_score}% ({state.validation.evidence_rating})
        
        Create a final consensus statement, state the final confidence rating percentage, list actionable clinical steps, and compile references.
        """
        system_prompt = "You are the clinical board director agent compiling the consensus report."
        consensus_state = self.llm.generate_json(prompt, system_prompt, ConsensusState)
        trace = f"Step 7 (Consensus Agent): Consensus achieved. Final report compiled. Confidence: {consensus_state.confidence_rating}."
        return {
            "consensus": consensus_state,
            "reasoning_trace": state.reasoning_trace + [trace]
        }

print("✅ All 7 specialized agents defined.")

In [ ]:
# =====================================================================
# 7. LANGGRAPH WORKFLOW ORCHESTRATION
# =====================================================================

class BioReasonWorkflow:
    """
    Defines the LangGraph orchestration state machine.
    Sequentially executes: Mutation -> Gene -> Pathway -> Literature -> Therapy -> Validation -> Consensus
    """
    
    def __init__(self, agents: BiomedicalAgents):
        self.agents = agents
        self.workflow = self._build_workflow()

    def _build_workflow(self) -> StateGraph:
        builder = StateGraph(BioReasonState)
        
        builder.add_node("mutation_node", self.agents.run_mutation_analysis)
        builder.add_node("gene_node", self.agents.run_gene_protein_impact)
        builder.add_node("pathway_node", self.agents.run_pathway_analysis)
        builder.add_node("literature_node", self.agents.run_literature_intelligence)
        builder.add_node("therapy_node", self.agents.run_therapy_matching)
        builder.add_node("validation_node", self.agents.run_evidence_validation)
        builder.add_node("consensus_node", self.agents.run_consensus_aggregation)
        
        builder.set_entry_point("mutation_node")
        builder.add_edge("mutation_node", "gene_node")
        builder.add_edge("gene_node", "pathway_node")
        builder.add_edge("pathway_node", "literature_node")
        builder.add_edge("literature_node", "therapy_node")
        builder.add_edge("therapy_node", "validation_node")
        builder.add_edge("validation_node", "consensus_node")
        builder.add_edge("consensus_node", END)
        
        return builder.compile()

    def execute(self, query: str) -> BioReasonState:
        initial_state = BioReasonState(
            mutation_query=query,
            reasoning_trace=[]
        )
        result = self.workflow.invoke(initial_state.model_dump())
        return BioReasonState.model_validate(result)

print("✅ LangGraph Workflow orchestration compiled.")

In [ ]:
# =====================================================================
# 8. AUDIO PROCESSOR FOR CLINICAL DICTATION & SPEECH SYNTHESIS
# =====================================================================

class AudioProcessor:
    """
    Handles clinical dictation and report speech synthesis (TTS).
    """
    
    def __init__(self, temp_dir: str = "data/audio_temp"):
        self.temp_dir = temp_dir
        os.makedirs(self.temp_dir, exist_ok=True)

    def synthesize_report(self, text: str) -> Optional[str]:
        try:
            clean_text = text.replace("**", "").replace("#", "").replace("-", " ")
            tts = gTTS(text=clean_text, lang='en', slow=False)
            output_path = os.path.join(self.temp_dir, "consensus_report.mp3")
            
            if os.path.exists(output_path):
                try:
                    os.remove(output_path)
                except Exception:
                    pass
                    
            tts.save(output_path)
            return output_path
        except Exception as e:
            print(f"Audio synthesis error: {e}")
            return None

print("✅ Audio Synthesis Processor registered.")

## 🚀 Run BioReason-X Causal Reasoning Pipeline

Enter a mutation query below, select the preferred LLM provider, and run the cells. 

By default, the `LLM_PROVIDER` environment variable is not set, meaning it runs in **Offline Mock Mode**, resolving variants deterministically with detailed annotations. If you have an **OpenAI API Key** or a local **vLLM model server**, you can configure them below!

In [ ]:
# =====================================================================
# 9. RUN PIPELINE AND VIEW INTERACTIVE RESULTS
# =====================================================================

# Optional configurations:
# os.environ["LLM_PROVIDER"] = "openai" # "openai" or "vllm"
# os.environ["OPENAI_API_KEY"] = "your-openai-api-key"

# 1. Select Mutation Target
mutation_query = "BRCA1 c.5266dupC"  # Try: "EGFR L858R", "BRCA1 c.5266dupC" or a custom one

print(f"🧬 Initializing BioReason-X for: '{mutation_query}'...\n")

# 2. Instantiate and compile components
llm = LLMClient()
fetcher = DataFetcher(cache_file_path="data/cached_mutations.json")
graph = BiomedicalKnowledgeGraph()
vector_store = BiomedicalVectorStore(persist_dir="data/chroma_db_notebook")
agents = BiomedicalAgents(llm, fetcher, graph, vector_store)
workflow = BioReasonWorkflow(agents)
audio_processor = AudioProcessor()

# 3. Execute LangGraph multi-agent loop
state = workflow.execute(mutation_query)

# 4. Display Chronological Trace Logs
print("\n" + "="*50)
print("⚙️ LANGEGRAPH MULTI-AGENT EXECUTION TRACE")
print("="*50)
for step in state.reasoning_trace:
    print(f"✅ {step}")
print("="*50 + "\n")

### 📈 Final Clinical Consensus Report

In [ ]:
# Format and display the final clinical consensus statement beautifully in markdown
consensus = state.consensus
ma = state.mutation_analysis
gp = state.gene_protein
th = state.therapy
val = state.validation

markdown_report = f"""
## 📄 Clinical Consensus Report: {mutation_query}

### 🧬 Variant Classification
- **Gene Symbol:** `{ma.gene_symbol}`
- **Variant Nomenclature:** `{ma.variant_nomenclature}`
- **Mutation Type:** `{ma.mutation_type}`
- **Pathogenicity (ClinVar):** `{ma.pathogenicity}`
- **Clinical Summary:** {ma.clinical_summary}

### 🔬 Protein & Sequence Impact
- **UniProt ID:** `{gp.uniprot_id}`
- **Protein Domains affected:** {gp.protein_domains}
- **Functional Consequence:** {gp.functional_consequence}
- **Molecular Mechanism:** {gp.molecular_mechanism}

### 🎯 Targeted Curation & Drug Recommendations
- **FDA-Approved / Investigational Therapies:** {', '.join([f'`{d}`' for d in th.recommended_therapies])}
- **Mechanism of Action (Rationale):** {th.mechanism_of_action}
- **Resistance & Escape Risks:** {th.resistance_risks}
- **Evidence Rating (Curation Board):** **{val.evidence_rating}** (Validation score: {val.validation_score}%)

### ⚖️ Final Consensus & Actionable Next Steps
- **Confidence Rating:** **{consensus.confidence_rating}**
- **Consensus Statement:** {consensus.final_consensus_statement}
- **Recommended Action Items:**
{chr(10).join([f'  - {action}' for action in consensus.recommended_actions])}

---
#### 📚 Key References
{chr(10).join([f'- 📄 {ref}' for ref in consensus.references])}
"""

display(Markdown(markdown_report))

### 🔊 Play Vocal Summary Report

In [ ]:
# Synthesize and play report audio directly in the notebook using IPython Audio widget
audio_path = audio_processor.synthesize_report(consensus.final_consensus_statement)

if audio_path and os.path.exists(audio_path):
    print("Vocal audio synthesized successfully. Play below:")
    display(Audio(audio_path, autoplay=False))
else:
    print("Audio synthesis was skipped or failed.")

### 🕸️ Interactive Pathway Network Graph
Below is the interactive spring-layout Plotly graph tracing the biology causal chain from the mutation through signaling pathways to FDA-approved drug recommendations.

In [ ]:
# Generate and show interactive Plotly figure inline in the notebook
fig = graph.generate_plotly_figure()
fig.show()